# Roopsee Profile Score Coverage Audit

This notebook uses the same scoring engine as the app. Scores are product-type aware, use the rounded average of applicable sheet-backed components, and preserve -100 as a hard blocker. It follows the full gender-free PnC grid: 8C1 skin profile * 14C1 concerns * (3C0 + 3C1 + 3C2 + 3C3 + explicit None) special states * 4 age states = 4,032 rows.

## Setup

Set `SCORE_WORKBOOK_PATH` and `PRODUCTS_CSV_PATH` before running, or place the files in `data/` using the default names from the README.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "roopsee_coverage").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from roopsee_coverage.constants import DEFAULT_PRODUCTS_CSV, DEFAULT_SCORE_WORKBOOK
from roopsee_coverage.engine import RecommendationEngine
from roopsee_coverage.profiles import all_profile_combinations

score_workbook = Path(os.getenv("SCORE_WORKBOOK_PATH", DEFAULT_SCORE_WORKBOOK)).expanduser()
products_csv = Path(os.getenv("PRODUCTS_CSV_PATH", DEFAULT_PRODUCTS_CSV)).expanduser()

engine = RecommendationEngine(score_workbook, products_csv)
engine.health()

## Coverage Counts By Full Valid Profile Grid

In [ ]:
profiles = all_profile_combinations(include_optional_age=True)
coverage_rows = engine.coverage_rows(profiles, top_n=3)

table_rows = []
for row in coverage_rows:
    profile = row["profile"]
    table_rows.append({
        "profile_id": row["profile_id"],
        "status": row["status"],
        "skin_type": profile["selectedSkinType"],
        "age": profile["age"],
        "gender": "removed",
        "face_body_concerns": ", ".join(profile["selectedFaceBodyConcerns"]),
        "lips_eyes_concerns": ", ".join(profile["selectedLipsEyesConcerns"]),
        "special_state": profile.get("specialConditionState", ", ".join(profile["selectedSpecialConditions"])),
        "scoring_special_conditions": ", ".join(row["profile"]["selectedSpecialConditions"]),
        "target_sheets": ", ".join(row["target_sheets"]),
        "total_matches": row["total_matches"],
        "products_above_90": row["above_90"],
        "products_above_80": row["above_80"],
        "products_above_70": row["above_70"],
        "products_above_60": row["above_60"],
        "products_above_50": row["above_50"],
        "top_products": " | ".join(f"{p['product_name']} ({p['score']})" for p in row["top_products"]),
    })

coverage_df = pd.DataFrame(table_rows)
coverage_df

## Quick Summary

In [ ]:
summary = pd.DataFrame([
    {"metric": "profiles", "value": len(coverage_df)},
    {"metric": "good_status_profiles", "value": int((coverage_df["status"] == "Strong").sum())},
    {"metric": "present_status_profiles", "value": int(coverage_df["status"].isin(["Usable", "Limited but usable"]).sum())},
    {"metric": "weak_status_profiles", "value": int((coverage_df["status"] == "Coverage gap").sum())},
    {"metric": "min_products_above_90", "value": int(coverage_df["products_above_90"].min())},
    {"metric": "min_products_above_80", "value": int(coverage_df["products_above_80"].min())},
    {"metric": "min_products_above_70", "value": int(coverage_df["products_above_70"].min())},
])
summary

## Weakest Profiles First

In [ ]:
coverage_df.sort_values(
    ["products_above_80", "products_above_70", "products_above_90"],
    ascending=[True, True, True],
).head(15)

## Optional Export

In [ ]:
output_path = PROJECT_ROOT / "profile_score_coverage_summary.csv"
coverage_df.to_csv(output_path, index=False)
output_path